### Import Necessary functions, Use the correct catalog and schema

In [0]:
from pyspark.sql import functions as F

spark.sql("use catalog project")
spark.sql("use schema bronze")

DataFrame[]

In [0]:
# Set the output path
OUTPUT_PATH = "project.silver"

In [0]:
df_customers = spark.table("bronze_customers")

# check for null values
null_exists = False
for col in df_customers.columns:
    if df_customers.filter(F.col(col).isNull()).select(col).count() > 0:
        null_exists = True
        break

print("="*70)
print("Null values exists" if null_exists else "Null values do not exist")
print("="*70)

display(df_customers.limit(5))


Null values do not exist


customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,ingestion_timestamp
06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP,2026-03-09T13:55:22.291Z
18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP,2026-03-09T13:55:22.291Z
4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP,2026-03-09T13:55:22.291Z
b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP,2026-03-09T13:55:22.291Z
4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP,2026-03-09T13:55:22.291Z


In [0]:
# calculating the total rows in the dataset
df_customers.count()

99441

### Checking the unique categorical values in state and city features

If the cardinality of these categorical features is excessively high, I will consider excluding them from the feature set to avoid inefficient encoding.

In [0]:
display(df_customers.groupBy("customer_state").count())
display(df_customers.groupBy("customer_city").count())

customer_state,count
SP,41746
SC,3637
MG,11635
PR,5045
RJ,12852
RS,5466
PA,975
GO,2020
ES,2033
BA,3380


customer_city,count
franca,161
sao bernardo do campo,938
sao paulo,15540
mogi das cruzes,383
campinas,1444
jaragua do sul,89
timoteo,54
curitiba,1521
belo horizonte,2773
montes claros,211


%md
## Observation: There are total 27 states and 4k+ cities

Droping the `customer_city`

In [0]:
df1 = df_customers.drop("customer_city")
display(df1.limit(3))

customer_id,customer_unique_id,customer_zip_code_prefix,customer_state,ingestion_timestamp
06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,SP,2026-03-09T13:55:22.291Z
18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,SP,2026-03-09T13:55:22.291Z
4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,SP,2026-03-09T13:55:22.291Z


In [0]:
from pyspark.sql.functions import col, trim, upper

df_customers_silver = (
    df1
    .withColumn("customer_state", upper(trim(col("customer_state"))))
    .dropDuplicates(["customer_unique_id"])
)

display(df_customers_silver.limit(5))

customer_id,customer_unique_id,customer_zip_code_prefix,customer_state,ingestion_timestamp
06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,SP,2026-03-09T13:55:22.291Z
18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,SP,2026-03-09T13:55:22.291Z
4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,SP,2026-03-09T13:55:22.291Z
b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,SP,2026-03-09T13:55:22.291Z
4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,SP,2026-03-09T13:55:22.291Z


In [0]:
df_customers_silver.write.format('delta').mode('overwrite').saveAsTable(f"{OUTPUT_PATH}.silver_customers")